In [ ]:
import pandas as pd
import numpy as np

# 1. FILES TO PROFILE
file_names = [
    "../data/raw/wuzzuf_jobs.csv",
    # Add more files here
]

# 2. MISSING VALUE DEFINITIONS
missing_values = [
    "",
    " ",
    "N/A",
    "NA",
    "n/a",
    "na",
    "None",
    "none",
    "NULL",
    "null",
    "-"
]

na_patterns = [
    "N/A",
    "NA",
    "n/a",
    "na",
    "None",
    "none",
    "NULL",
    "null",
    "-"
]


 
# 3. FUNCTION TO PROFILE ONE FILE
def profile_file(file_path):

    print("\n")
    print("=" * 100)
    print(f"FILE: {file_path}")
    print("=" * 100)

    # Load data
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"ERROR reading file: {e}")
        return

    # Basic information
    row_count, column_count = df.shape
    duplicate_rows = df.duplicated().sum()

    print("\n--- DATASET SUMMARY ---")

    print(f"Rows            : {row_count:,}")
    print(f"Columns         : {column_count:,}")
    print(f"Duplicate Rows  : {duplicate_rows:,}")

    # Column names
    print("\n--- COLUMN NAMES ---")

    for i, column in enumerate(df.columns, start=1):
        print(f"{i}. {column}")

    # Prepare data for profiling
    df_profile = df.copy()

    df_profile = df_profile.replace(
        missing_values,
        np.nan
    )

    # Create profile
    profile = pd.DataFrame()

    profile["column_name"] = df_profile.columns

    # Data type
    profile["data_type"] = [
        str(df_profile[col].dtype)
        for col in df_profile.columns
    ]

    # Row count
    profile["row_count"] = row_count

    # Null count
    profile["null_count"] = [
        df_profile[col].isna().sum()
        for col in df_profile.columns
    ]

    # Null percentage
    profile["null_percentage"] = [
        round(
            df_profile[col].isna().mean() * 100,
            2
        )
        for col in df_profile.columns
    ]

    # Unique values
    profile["unique_count"] = [
        df_profile[col].nunique(dropna=True)
        for col in df_profile.columns
    ]

    # Duplicate values
    profile["duplicate_count"] = [
        df_profile[col].duplicated().sum()
        for col in df_profile.columns
    ]

    # N/A count
    na_count_list = []

    for col in df.columns:

        count = (
            df[col]
            .astype(str)
            .str.strip()
            .isin(na_patterns)
            .sum()
        )

        na_count_list.append(count)

    profile["na_count"] = na_count_list

    # Empty string count
    empty_count = []

    for col in df.columns:

        count = (
            df[col]
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )

        empty_count.append(count)

    profile["empty_string_count"] = empty_count

    # Sample values
    sample_values = []

    for col in df_profile.columns:

        values = (
            df_profile[col]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(5)
            .tolist()
        )

        sample_values.append(
            " | ".join(values)
        )

    profile["sample_values"] = sample_values

    # Min / Max for numeric columns
    min_values = []
    max_values = []

    for col in df_profile.columns:

        if pd.api.types.is_numeric_dtype(
            df_profile[col]
        ):

            min_values.append(
                df_profile[col].min()
            )

            max_values.append(
                df_profile[col].max()
            )

        else:

            min_values.append("")
            max_values.append("")

    profile["min_value"] = min_values
    profile["max_value"] = max_values

    # Most common value
    most_common = []
    most_common_count = []

    for col in df_profile.columns:

        value_counts = (
            df_profile[col]
            .value_counts(dropna=True)
        )

        if len(value_counts) > 0:

            most_common.append(
                str(value_counts.index[0])
            )

            most_common_count.append(
                int(value_counts.iloc[0])
            )

        else:

            most_common.append("")
            most_common_count.append(0)

    profile["most_common_value"] = most_common
    profile["most_common_count"] = most_common_count

    # PRINT COLUMN PROFILE
    print("\n--- COLUMN PROFILE ---\n")

    print(
        profile.to_string(index=False)
    )

    # PRINT SAMPLE ROWS
    print("\n--- SAMPLE ROWS ---\n")

    sample_rows = df_profile.sample(
        min(10, len(df_profile)),
        random_state=42
    )

    print(
        sample_rows.to_string(index=False)
    )

    # PRINT UNIQUE VALUES
    print("\n--- UNIQUE VALUES PER COLUMN ---")

    for col in df_profile.columns:

        unique_values = (
            df_profile[col]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(20)
            .tolist()
        )

        print(f"\n{col}")
        print(f"Unique count: {df_profile[col].nunique(dropna=True)}")
        print(
            "Values:",
            " | ".join(unique_values)
        )

    print("\n" + "=" * 100)
    print(f"FINISHED: {file_path}")
    print("=" * 100)


 
# 4. PROFILE ALL FILES
for file_name in file_names:

    profile_file(file_name)

print("\n")
print("=" * 100)
print("ALL FILES COMPLETED")
print("=" * 100)



FILE: ../data/raw/wuzzuf_jobs.csv

--- DATASET SUMMARY ---
Rows            : 88
Columns         : 18
Duplicate Rows  : 0

--- COLUMN NAMES ---
1. job_id
2. source
3. job_title
4. company_name
5. job_description
6. job_url
7. country
8. city
9. work_mode
10. employment_type
11. career_level
12. experience_needed
13. qualification_required
14. skills
15. salary
16. estimated_salary
17. posted_date
18. collected_at

--- COLUMN PROFILE ---

           column_name data_type  row_count  null_count  null_percentage  unique_count  duplicate_count  na_count  empty_string_count                                                                                                                                                                                                                                                                                                                                                                                                                                        